If you're opening this Notebook on colab, you will probably need to install 🤗 Transformers and 🤗 Datasets. Uncomment the following cell and run it.

In [ ]:
! pip install datasets transformers
#!pip install datasets==4.8.4
#!pip install transformers
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


If you're opening this notebook locally, make sure your environment has an install from the last version of those libraries.

In [ ]:
#from huggingface_hub import notebook_login

#notebook_login()

In [ ]:
#!apt install git-lfs
!pip install torch

In [ ]:
!pip install av
!pip install torchcodec
!pip install av torchcodec
!pip install torchvision

# 1. Install specific stable PyAV version and Torchcodec
#!pip install av==11.0.0 torchcodec datasets accelerate --upgrade

# 2. Force an update to the torchvision package if backend integration is stuck
#!pip install torchvision --upgrade --no-cache-dir

In [ ]:
#!pip install --force-reinstall torchaudio --index-url https://pytorch.org
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch torchvision torchaudio --index-url https://pytorch.org

In [ ]:
from torchcodec.decoders import VideoDecoder
import sys
import torchvision.io

# Create a dummy object to satisfy the internal import check
class DummyVideoReader:
    pass

torchvision.io.VideoReader = DummyVideoReader
sys.modules['torchvision.io'].VideoReader = DummyVideoReader


running above block needed to successfully import videoDecoder in next block(avoids:- "Failed: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)" error)

In [ ]:
import os
# Force torchvision to look for the PyAV backend explicitly
os.environ["TORCHVISION_VIDEO_BACKEND"] = "pyav"

import torch
import torchvision
from datasets import Dataset

# Verify the class is now safely bound into the namespace
try:
    from torchvision.io import VideoReader
    print("Success: VideoReader successfully imported!")
except ImportError as e:
    print(f"Failed: {e}")

Success: VideoReader successfully imported!


In [ ]:
import torch
print("GPU Connected:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Connected: False


In [ ]:
from torchcodec.decoders import VideoDecoder

Make sure your version of Transformers is at least 4.11.0 since the functionality was introduced in that version:

In [ ]:
import transformers

print(transformers.__version__)

5.13.1


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import os

print(os.getcwd())

path ="/content/gdrive/MyDrive/"
print(os.listdir(path))

/content
['analytics_vidhya', 'BERT_Model', 'Colab Notebooks', 'creditvidya_model_folder', 'Talentica_ML', 'weather_forcast', 'zenatic_ac_model_folders', 'AC_Data_hour_with_temp1.xlsx', 'AC_Data_hour_with_temp.xlsx', 'AC_Data_hour.xlsx', 'AC_Data.csv', 'hiring_assignment_cv.xlsx', 'python_intro.ipynb', 'stock_forcast_model.h5', 'test_new_6thsense.csv', 'ner_dataset.csv', 'Large_videos_pics_originaldrive', 'upsc_2026_speciesInNews.docx', 'Large_videos_part2', 'distilgpt2-finetuned-wikitext2', 'distilbert-base-uncased-finetuned-wikitext2']


You can find a script version of this notebook to fine-tune your model in a distributed fashion using multiple GPUs or TPUs [here](https://github.com/huggingface/transformers/tree/master/examples/language-modeling).

# Fine-tuning a language model

In this notebook, we'll see how to fine-tune 🤗 masked transformers based model for NLP. We'll train mode by **trainer wrapper** as well as **accelerator loop(latter for more customization)**. So what are masked models?

- **Masked language modeling**: Masked models are **class of encoder based(discriminative) model**. The model has to predict some tokens that are masked in the input(although their actual value present in labels/output set i.e. Y). It still has access to the whole sentence, so it can use the tokens before and after the tokens masked to predict their value.


## Preparing the dataset

For each of those tasks, we will use the [Wikitext 2]() dataset as an example. You can load it very easily with the 🤗 Datasets library.

In [ ]:
from datasets import load_dataset
datasets = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

You can replace the dataset above with any dataset hosted on [the hub](https://huggingface.co/datasets) (🤗) or use your own files. Just uncomment the following cell and replace the paths with values that will lead to your files:

In [ ]:
# datasets = load_dataset("text", data_files={"train": path_to_train.txt, "validation": path_to_validation.txt}

You can also load datasets from a csv or a JSON file, see the [full documentation](https://huggingface.co/docs/datasets/loading_datasets.html#from-local-files) for more information.

To access an actual element, you need to select a split first, then give an index:

In [ ]:
datasets["train"][1]

{'text': ' = Valkyria Chronicles III = \n'}

In [ ]:
type(datasets), datasets.keys(), type(datasets["train"]), len(datasets["train"]), type(datasets["train"][:3]), datasets["train"][10].keys()

(datasets.dataset_dict.DatasetDict,
 dict_keys(['test', 'train', 'validation']),
 datasets.arrow_dataset.Dataset,
 36718,
 dict,
 dict_keys(['text']))

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset.

In [ ]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(datasets["train"])

,text
0,"Jesus emphasized the need for pure thoughts as well as actions , and stated , "" Everyone who looks at a woman lustfully has already committed adultery with her in his heart "" ( Matthew 5 : 28 ) . The Catechism states that , with the help of God 's grace , men and women are required to overcome lust and bodily desires "" for sinful relationships with another person 's spouse . "" In Theology of the Body , a series of lectures given by Pope John Paul II , Jesus ' statement in Matthew 5 : 28 is interpreted that one can commit adultery in the heart not only with another 's spouse , but also with his / her own spouse if one looks at him / her lustfully or treats him / her "" only as an object to satisfy instinct "" . \n"
1,"In the courtroom trials , the player aims to get their client declared "" not guilty "" . To do so , they cross @-@ examine witnesses , and aim to find lies and inconsistencies in the testimonies . They are able to go back and forth between the different statements in the testimony , and can press the witness for more details on a statement . When the player finds an inconsistency , they can present a piece of evidence that contradicts the statement . The player is penalized if they present incorrect evidence : in the first game , a number of exclamation marks is shown , with one disappearing after each mistake the player makes ; in later games , a health bar that represents the judge 's patience is used instead . If all exclamation marks are lost , or the health bar reaches zero , the player loses the game and their client is declared guilty . \n"
2,= = Family and early life = = \n
3,= = Major intersections = = \n
4,
5,"While Leigh made Streetcar in 1951 , Olivier joined her in Hollywood to film Carrie , based on the controversial novel Sister Carrie ; although the film was plagued by troubles , Olivier received warm reviews and a BAFTA nomination . Olivier began to notice a change in Leigh 's behaviour , and he later recounted that "" I would find Vivien sitting on the corner of the bed , wringing her hands and sobbing , in a state of grave distress ; I would naturally try desperately to give her some comfort , but for some time she would be inconsolable . "" After a holiday with Coward in Jamaica , she seemed to have recovered , but Olivier later recorded , "" I am sure that ... [ the doctors ] must have taken some pains to tell me what was wrong with my wife ; that her disease was called manic depression and what that meant — a possibly permanent cyclical to @-@ and @-@ fro between the depths of depression and wild , uncontrollable mania . He also recounted the years of problems he had experienced because of Leigh 's illness , writing , "" throughout her possession by that uncannily evil monster , manic depression , with its deadly ever @-@ tightening spirals , she retained her own individual canniness — an ability to disguise her true mental condition from almost all except me , for whom she could hardly be expected to take the trouble . "" \n"
6,
7,"In the past few years , lots of training exercises took place in Romania with other Balkan or Allied countries . Most of these exercises took place at Babadag , which is one of the largest and most modern training firing ranges and military facilities in Europe , with a total surface area of 270 square kilometres . It was announced on December 6 , 2006 that 1 @,@ 500 U.S. troops stationed at Mihail Kogălniceanu , which in time will form Joint Task Force East , will be using Babadag as a training base . \n"
8,= = = Unveiling = = = \n
9,"Eguchi had often rushed when drawing his earlier manga Susume ! ! Pirates ( すすめ ! ! パイレーツ ) , but starting with Stop ! ! Hibari @-@ kun ! , he raised the standards he held for his art and had to began taking more time to draw the chapters . In addition , Eguchi was very particular about the appearance of his manuscripts , so he never used white @-@ out to correct any drawing errors because he disliked how it looked . A

As we can see, some of the texts are a full paragraph of a Wikipedia article while others are just titles or empty lines.

# **===============================================================================**

## Masked language modeling

For masked language modeling (MLM) we are going to use the same preprocessing as before for our dataset with one additional step: we will randomly mask some tokens (by replacing them by `[MASK]`) and the labels will be adjusted to only include the masked tokens (we don't have to predict the non-masked tokens).

We will use the [`distilroberta-base`](https://huggingface.co/distilroberta-base) model for this example. You can pick any of the checkpoints listed [here](https://huggingface.co/models?filter=masked-lm) instead:

In [ ]:
#model_checkpoint = "distilroberta-base"
model_checkpoint = "distilbert-base-uncased"

We can now call the tokenizer on all our texts. This is very simple, using the map method from the Datasets library. First we define a function that call the tokenizer on our texts:

We’ll also grab the word IDs if they are available, as we will need them later on to do whole word masking. We’ll wrap this in a simple function

In [ ]:
def tokenize_function(examples):
    #result = tokenizer(examples["text"], truncation=True) # beacuse of sequence lengths>512 producing index error
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

We can apply the same tokenization function as before(after importing tokenizer), we just need to update our tokenizer to use the checkpoint we just picked and check for pad tokens if blocks length is uneven:

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Check if pad_token is already assigned; if not, set it to the eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
tokenized_datasets = datasets.map(tokenize_function, batched=True, num_proc=8, remove_columns=["text"])

Map (num_proc=8):   0%|          | 0/4358 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


Map (num_proc=8):   0%|          | 0/36718 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (552 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (515 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (844 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (647 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (608 > 512). Running this sequence through the model will result in indexing errors
[transformers] 

Map (num_proc=8):   0%|          | 0/3760 [00:00<?, ? examples/s]

If we now look at an element of our datasets, we will see the text have been replaced by the `input_ids` the model will need:

In [ ]:
print(tokenized_datasets["train"][1])
print(type(tokenized_datasets["train"]))
tokenizer.decode(tokenized_datasets["train"][1])

{'input_ids': [101, 1027, 11748, 4801, 4360, 11906, 3523, 1027, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1], 'word_ids': [None, 0, 1, 1, 1, 2, 3, 4, None]}
<class 'datasets.arrow_dataset.Dataset'>


'[CLS] = valkyria chronicles iii = [SEP]'

We group/concat texts together and chunk them in samples of equal length `block_size`.
To do this, we will use the `map` method again, with the option `batched=True` along wit finite `batch size`(after defining **group_text** fn).

This option actually lets us change the number of examples in the datasets by returning a different number of examples than we got. This way, we can create our new samples from a batch of examples.

First, we grab the maximum length our model was pretrained with. This might be a big too big to fit in your GPU RAM, so here we take a bit less at just 128.

You can skip that step if your dataset is composed of individual sentences.

In [ ]:
# block_size = tokenizer.model_max_length
block_size = 128

In [ ]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,# deafualt batch_szie also 1000
    num_proc=8,
)# here batch inside map only facilitates faster processing, does not group final data in batches like pytorch dataloader(so keep batch size =32, 64 in dataloader)

Map (num_proc=8):   0%|          | 0/4358 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/36718 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
print(type(lm_datasets["train"]), type(lm_datasets["validation"]))
print(len(lm_datasets["train"]), len(lm_datasets["validation"]), len(lm_datasets["test"]))

<class 'datasets.arrow_dataset.Dataset'> <class 'datasets.arrow_dataset.Dataset'>
18551 1918 2198


And we can check our datasets have changed: now the samples contain chunks of `block_size` contiguous tokens, potentially spanning over several of our original texts. Also Input ids = labels

In [ ]:
print(tokenizer.decode(lm_datasets["train"][1]["input_ids"]))
if(tokenizer.decode(lm_datasets["train"][1]["input_ids"]) == tokenizer.decode(lm_datasets["train"][1]["labels"])):
  print("Same")

the first game and follows the " nameless ", a penal military unit serving the nation of gallia during the second europan war who perform secret black operations and are pitted against the imperial unit " calamaty raven ". [SEP] [CLS] the game began development in 2010, carrying over a large portion of the work done on valkyria chronicles ii. while it retained the standard features of the series, it also underwent multiple adjustments, such as making the game more forgiving for series newcomers. character designer raita honjou and composer hitoshi sakimoto both returned from previous entries, along with valkyria chronicles ii director takeshi
Same


In [ ]:
#from pprint import pprint
for iter in range(5):
  elm_inputid = lm_datasets["train"][iter]["input_ids"]
  if(-100 not in elm_inputid):
    print(elm_inputid)
    print("Padded", len(elm_inputid))

  print("~"*100)
  elm_mask =lm_datasets["train"][iter]["attention_mask"]
  if(0 not in elm_mask):
    print(elm_mask)
    print("complete_masked", len(elm_mask))
  print("-"*270)

[101, 102, 101, 1027, 11748, 4801, 4360, 11906, 3523, 1027, 102, 101, 102, 101, 12411, 5558, 2053, 11748, 4801, 4360, 1017, 1024, 4895, 2890, 27108, 5732, 11906, 1006, 2887, 1024, 1856, 1806, 1671, 30222, 30218, 30259, 30227, 30255, 30258, 30219, 2509, 1010, 5507, 1012, 11748, 4801, 4360, 1997, 1996, 11686, 1017, 1007, 1010, 4141, 3615, 2000, 2004, 11748, 4801, 4360, 11906, 3523, 2648, 2900, 1010, 2003, 1037, 8608, 2535, 1030, 1011, 1030, 2652, 2678, 2208, 2764, 2011, 16562, 1998, 2865, 1012, 4432, 2005, 1996, 9160, 12109, 1012, 2207, 1999, 2254, 2249, 1999, 2900, 1010, 2009, 2003, 1996, 2353, 2208, 1999, 1996, 11748, 4801, 4360, 2186, 1012, 15440, 1996, 2168, 10077, 1997, 8608, 1998, 2613, 1030, 1011, 1030, 2051, 11247, 2004, 2049, 16372, 1010, 1996, 2466, 3216, 5903, 2000]
Padded 128
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
lm_datasets.keys(), lm_datasets["train"][1].keys()

(dict_keys(['test', 'train', 'validation']),
 dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'word_ids', 'labels']))

The rest is very similar to what we had, with two exceptions. First we use a model suitable for masked LM:

In [ ]:
from transformers import AutoModelForMaskedLM
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch(when loading **distilroberta**). But this error not there when using **distilbert** in above block

# Training approaches:-(1)using .trainer() method, (2) accelerator loop

**Loss Selection**
Choose Internal Model Loss (AutoCausalLM / Forward Pass Loss): Let the AutoModelForCausalLM compute the cross-entropy loss internally by passing labels=input_ids (shifted internally by the model).

**Why:** Computing loss inside the model ensures proper handling of shifted next-token prediction targets and keeps mixed-precision scaling stable under Accelerate. Manual cross-entropy calculation outside the model requires flattening logits and shifting tokens manually, risking shape mismatch errors or full-precision bottlenecks.

We redefine our `TrainingArguments`:

*1)Trainer method*

In [ ]:
from transformers import Trainer, TrainingArguments

Like before, the last argument to setup everything so we can push the model to the [Hub](https://huggingface.co/models) regularly during training. Remove it if you didn't follow the installation steps at the top of the notebook. If you want to save your model locally in a name that is different than the name of the repository it will be pushed, or if you want to push your model under an organization and not your name space, use the `hub_model_id` argument to set the repo name (it needs to be the full name, including your namespace: for instance `"sgugger/bert-finetuned-wikitext2"` or `"huggingface/bert-finetuned-wikitext2"`).

Finally, we use a special `data_collator`. The `data_collator` is a function that is responsible of taking the samples and batching them in tensors. In the previous example, we had nothing special to do, so we just used the default for this argument. Here we want to do the random-masking. We could do it as a pre-processing step (like the tokenization) but then the tokens would always be masked the same way at each epoch. By doing this step inside the `data_collator`, we ensure this random masking is done in a new way each time we go over the data.

To do this masking for us, the library provides a `DataCollatorForLanguageModeling`. We can adjust the probability of the masking:

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

To see how the random masking works, let’s feed a few examples to the data collator. Since it expects a list of dicts, where each dict represents a single chunk of contiguous text, we first iterate over the dataset before feeding the batch to the collator. We remove the "word_ids" key for this data collator as it does not expect it:

In [ ]:
samples = [lm_datasets["train"][i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(f"\n'>>> {tokenizer.decode(chunk)}'")


'>>> [CLS] [SEP] [CLS] = [MASK]kyria chronicles iii = [SEP] [CLS] [SEP] [CLS] senjo no valkyria 3 : unre [MASK]ded chronicles ( japanese : 戦 場 のウァルnumュリア3 [MASK] lit. [MASK] [MASK]ria of the battlefield 3 ), commonly referred [MASK] as valkyria chronicles iii outside japan, is a tactical role [MASK] - @ playing video [MASK] developed by sega and [MASK]. vision for the playstation portable. released in january [MASK] in [MASK], it is [MASK] third game in the val [MASK]ria series. [MASK] the same fusion of tactical and real @ - @ time gameplay as its predecessors, the story runs parallel to'

'>>> the first game and [MASK] the " nameless [MASK], a [MASK] military unit serving the nation of gallia [MASK] [MASK] second europa [MASK] war who perform secret black operations and are [MASK] against the imperial unit [MASK] [MASK]amaty [MASK] ". [SEP] [CLS] the game began development in 2010 [MASK] carrying over a highlighted portion [MASK] the work done on valkyria chronicles ii. while it ret

concept of **whole_word_masking_data_collator** to be checked later in this colab

Our data already segregated in train, validation, test; else we can use follwoing code for train:test split(presently commented)

In [ ]:
"""
train_size = 10_000(size)
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset
"""

'\ntrain_size = 10_000(size)\ntest_size = int(0.1 * train_size)\n\ndownsampled_dataset = lm_datasets["train"].train_test_split(\n    train_size=train_size, test_size=test_size, seed=42\n)\ndownsampled_dataset\n'

And some `TrainingArguments`:

# here we use train and evaluation via batch with logs in every batch iter(something missed in fine-tuning causal model via trainer earlier)

In [ ]:
batch_size = 32
# Show the training loss with every epoch
logging_steps = len(lm_datasets["train"]) // batch_size
model_name = model_checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir= "./local_checkpoints"+f"{model_name}-finetuned-wikitext2",#avoid giving mounted drive path as models
    #will consumeheavy write operations that quickly max out your Drive storage and API limits
    eval_strategy = "epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
    fp16=True,
    save_strategy="epoch",
    #save_steps=1000,          # Don't save too frequently, but to be used when both eval strategy and save strategy are "steps"
    save_total_limit=1,      # Automatically deletes older checkpoints; keeps only the newest
    load_best_model_at_end=True, # Keeps your optimal weights

    per_device_train_batch_size=batch_size,# batch_wise train like in acclerator loop (not in causal model finetuning trainer method)
    per_device_eval_batch_size=batch_size,# batch_wise evaluation like in acclerator loop (not in causal model finetuning trainer method)
    logging_steps=logging_steps,# to show logs with each bactch iter(not in causal model finetuning trainer method)
)

Then we just have to pass everything to `Trainer` and begin training:

In [ ]:
'''
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)
'''

'\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=lm_datasets["train"],\n    eval_dataset=lm_datasets["validation"],\n    data_collator=data_collator,\n)\n'

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator= data_collator,
    processing_class= tokenizer,#The "tokenizer" argument in Hugging Face Trainer is deprecated, and processing_class is its modern replacement.
    #tokenizer = tokenizer,
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.233062,2.046615
2,2.096937,1.962038
3,2.044532,1.979630


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vocab_projector.weight'].


TrainOutput(global_step=1740, training_loss=2.1247746363453484, metrics={'train_runtime': 324.6466, 'train_samples_per_second': 171.426, 'train_steps_per_second': 5.36, 'total_flos': 1844356505052672.0, 'train_loss': 2.1247746363453484, 'epoch': 3.0})

Like before, we can evaluate our model on the validation set. The perplexity is much lower than for the CLM objective because for the MLM objective, we only have to make predictions for the masked tokens (which represent 15% of the total here) while having access to the rest of the tokens. It's thus an easier task for the model.

In [ ]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
2.044532,1.976085,3


Perplexity: 7.21


2)Using accelerator loop

In [ ]:
#import torch
#import torchvision
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, default_data_collator
from accelerate import Accelerator,DataLoaderConfiguration
from torch.optim import AdamW
from datasets import load_dataset

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

**DataCollatorForLanguageModeling also applies random masking with each evaluation**, so we’ll see some fluctuations in our perplexity scores with each training run. One way to eliminate this source of randomness is to apply the masking once on the whole test set, and then use the default data collator in 🤗 Transformers to collect the batches during evaluation. To see how this works, let’s implement a simple function that applies the masking on a batch, similar to our first encounter with DataCollatorForLanguageModeling:

In [ ]:
def insert_random_mask(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    # Create a new "masked" column for each column in the dataset
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

Next, we’ll apply this function to our test set and drop the **unmasked columns so we can replace them with the masked ones**. You can use whole word masking by replacing the data_collator above with the appropriate one, in which case you should remove the first line here:

In [ ]:
lm_datasets = lm_datasets.remove_columns(["word_ids"])

eval_dataset = lm_datasets["validation"].map(
    insert_random_mask,
    batched=True,
    remove_columns=lm_datasets["validation"].column_names,
)
eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_token_type_ids": "token_type_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

test_dataset = lm_datasets["test"].map(
    insert_random_mask,
    batched=True,
    remove_columns=lm_datasets["test"].column_names,
)
test_dataset = test_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_token_type_ids": "token_type_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

Map:   0%|          | 0/1918 [00:00<?, ? examples/s]

Map:   0%|          | 0/2198 [00:00<?, ? examples/s]

In [ ]:
#from torch.utils.data import DataLoader
#from transformers import default_data_collator

batch_size = 32
train_dataloader = DataLoader(
    lm_datasets["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)


In [ ]:
eval_dataloader = DataLoader(
    eval_dataset, batch_size=batch_size, collate_fn=default_data_collator
)# by default shuffle is false


test_dataloader = DataLoader(
    test_dataset, batch_size=batch_size, collate_fn=default_data_collator
)# by default shuffle is false

In [ ]:
print(len(lm_datasets["train"]), len(train_dataloader))
print(len(eval_dataset), len(eval_dataloader))
print(len(test_dataset), len(test_dataloader))

print(type(train_dataloader), type(lm_datasets["train"]))

18551 580
1918 60
2198 69
<class 'accelerate.data_loader.DataLoaderShard'> <class 'datasets.arrow_dataset.Dataset'>


In [ ]:
test_dataloader.keys()

AttributeError: 'StatefulDataLoader' object has no attribute 'keys'

In [ ]:
test_dataloader[0]

TypeError: 'DataLoaderShard' object is not subscriptable

In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=3e-5,
    eps=1e-8,
)

In [ ]:
'''
accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)
'''
# 1. Enable stateful dataloader configuration if tracking iteration states
dataloader_config = DataLoaderConfiguration(use_stateful_dataloader=True)
accelerator = Accelerator(dataloader_config=dataloader_config)

# 5. Prepare everything with Accelerator (crucial for distributed setups)
model, optimizer, train_dataloader, eval_dataloader, test_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader, test_dataloader)

In [ ]:

from transformers import get_linear_schedule_with_warmup

num_train_epochs = 3
#max_grad_norm = 1.0

# Total number of training steps is number of batches * number of epochs.
total_steps = len(train_dataloader) * num_train_epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [ ]:
from tqdm.auto import tqdm
import torch
import math

progress_bar = tqdm(range(total_steps))

train_loss_values, eval_loss_values = [], []

for epoch in range(num_train_epochs):
    # Put the model into training mode.
    model.train()

    # Reset the total loss for this epoch.
    total_loss = 0

    for batch in train_dataloader:
        outputs = model(**batch)

        # get the loss
        loss = outputs.loss

        # Perform a backward pass to calculate the gradients
        accelerator.backward(loss)

        # track train loss
        total_loss+= loss.item()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Calculate the average loss over the training data.
    avg_train_loss = total_loss / len(train_dataloader)
    print(f">>> Epoch {epoch}: avg_Training/train Loss: {avg_train_loss}")

    # Store the loss value for plotting the learning curve.
    train_loss_values.append(avg_train_loss)

    # Evaluation
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        loss = outputs.loss
        losses.append(accelerator.gather(loss.repeat(batch_size)))

    losses = torch.cat(losses)
    losses = losses[: len(eval_dataloader)]

    eval_loss = losses.mean().item()
    print(f">>> Epoch {epoch}: avg_Validation/eval Loss: {eval_loss}")

    # Store the loss value for plotting the learning curve.
    eval_loss_values.append(losses.mean().item())

    try:
        perplexity = math.exp(torch.mean(losses))
    except OverflowError:
        perplexity = float("inf")

    print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

  0%|          | 0/1740 [00:00<?, ?it/s]

>>> Epoch 0: avg_Training/train Loss: 2.0317559289521183
>>> Epoch 0: avg_Validation/eval Loss: 1.6698743104934692
>>> Epoch 0: Perplexity: 5.311500155390793
>>> Epoch 1: avg_Training/train Loss: 1.9797649950816714
>>> Epoch 1: avg_Validation/eval Loss: 1.6396726369857788
>>> Epoch 1: Perplexity: 5.153482176605042
>>> Epoch 2: avg_Training/train Loss: 1.9443689040068923
>>> Epoch 2: avg_Validation/eval Loss: 1.632906198501587
>>> Epoch 2: Perplexity: 5.118729166127649


In [ ]:
model.eval()
losses = []
for step, batch in enumerate(eval_dataloader):
    with torch.no_grad():
        outputs = model(**batch)

    loss = outputs.loss
    losses.append(accelerator.gather(loss.repeat(batch_size)))

losses = torch.cat(losses)
losses = losses[: len(eval_dataloader)]

eval_loss = losses.mean().item()
print(f">>> Epoch {epoch}: avg_Validation/eval Loss: {eval_loss}")

# Store the loss value for plotting the learning curve.
eval_loss_values.append(losses.mean().item())

try:
    perplexity = math.exp(torch.mean(losses))
except OverflowError:
    perplexity = float("inf")

print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

>>> Epoch 2: avg_Validation/eval Loss: 1.632906198501587
>>> Epoch 2: Perplexity: 5.118729166127649


# Use save method of whichever approach gives lower perplexity

**Save by Trainer method**

You can now upload the result of the training to the Hub, just execute this instruction:

In [ ]:
#trainer.push_to_hub()

In [ ]:
trainer.save_model(path+ f"{model_name}-finetuned-wikitext2")# also saves training_args.bin(not saved in accelerator method below)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

**save by acclereate loop method**

Your test data loader's batch size or length is **becoming zero from batch_size(32 here) because** ❎


*   Hugging Face Accelerate wrapper or distributed sampler splits or exhausts the
dataset across processes
*   or an active dynamic batch-size finder/callback scaled it down to zero

*   or the remaining data samples did not divide evenly into the number of active GPU

To fix these:-
1) re-initialize the dataloader(if possible)
2) retrieve original batch size instead of querying it from dynamic object(here dataloader)
3)or the remaining data samples did not divide evenly into the number of active GPU

2nd most practical

In [ ]:
print(test_dataloader.batch_size)
print(test_dataloader.collate_fn)
print(test_dataloader.num_workers)
print(test_dataloader.pin_memory)

print("*"*50)
print(train_dataloader.batch_size)
print(train_dataloader.collate_fn)
print(train_dataloader.num_workers)
print(train_dataloader.pin_memory)

print("*"*50)

print(eval_dataloader.batch_size)
print(eval_dataloader.collate_fn)
print(eval_dataloader.num_workers)
print(eval_dataloader.pin_memory)

None
<function default_data_collator at 0x7fd317e094e0>
0
False
**************************************************
None
DataCollatorForLanguageModeling(tokenizer=BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, wh

In [ ]:
# 1. Wait for all processes to finish training(Ensures no hanging or broken shards across multi-GPU setups)
accelerator.wait_for_everyone()# Synchronize processes before saving

# 2. Unwrap the model to strip away the distributed training shell
unwrapped_model = accelerator.unwrap_model(model)#Unwrap Model: accelerator.unwrap_model(model) removes the extra layers added by
#accelerator.prepare() so you get your raw base model back.

# 3. Save only on the main process to avoid file conflicts
if accelerator.is_main_process:
    # This creates the weights and the required config.json file

    model_name = model_checkpoint.split("/")[-1]
    unwrapped_model.save_pretrained(path+ f"{model_name}-finetuned-wikitext2", save_function=accelerator.save)#Save Function: save_function=accelerator.save makes
    #sure the model saves correctly across multiple GPUs or machines without breaking
    tokenizer.save_pretrained(path+ f"{model_name}-finetuned-wikitext2")

    # Save the test dataloader setup
    # Because DataLoaders are runtime iterators, we save the underlying dataset indices/tensors(here its test_dataset)
    dataloader_state = {
        #"dataset_dict": tokenized_datasets["test"].to_dict(), # Converts Apache Arrow table data to a dict
        #"dataset_dict": lm_datasets["test"].to_dict(),
        "dataset_dict": test_dataset.to_dict(),
        #"batch_size": test_dataloader.batch_size,#refer comment on 2nd last block
        "batch_size": batch_size,
        "collate_fn": test_dataloader.collate_fn,

        "num_workers": test_dataloader.num_workers,
        "pin_memory": test_dataloader.pin_memory,
    }
    accelerator.save(dataloader_state, os.path.join(path+ f"{model_name}-finetuned-wikitext2", "test_dataloader_state.pt"))
    print(f"Model and evaluation states successfully saved to {path+ f"{model_name}-finetuned-wikitext2"}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and evaluation states successfully saved to /content/gdrive/MyDrive/distilbert-base-uncased-finetuned-wikitext2


# Load saved model for inference

You can now share this model with all your friends, family, favorite pets: they can all load it with the identifier `"your-username/the-name-you-picked"` so for instance:

```python
from transformers import AutoModelForMaskedLM

model = AutoModelForMaskedLM.from_pretrained("sgugger/my-awesome-model")
```

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import Dataset
from accelerate import Accelerator, DataLoaderConfiguration
import os

In [ ]:
#from transformers import AutoTokenizer, AutoModelForMaskedLM

path ="/content/gdrive/MyDrive/"

#model_checkpoint = "distilroberta-base"
model_checkpoint = "distilbert-base-uncased"

model_name = model_checkpoint.split("/")[-1]

# 1. Enable stateful dataloader configuration if tracking iteration states(states of dataloader)
dataloader_config = DataLoaderConfiguration(use_stateful_dataloader=True)
accelerator = Accelerator(dataloader_config=dataloader_config)

# 1. Reconstruct the saved Model and Tokenizer
tokenizer_infer =AutoTokenizer.from_pretrained(path+ f"{model_name}-finetuned-wikitext2", use_fast=True)
model_infer = AutoModelForMaskedLM.from_pretrained(path+ f"{model_name}-finetuned-wikitext2")

# 2. Reconstruct the saved Test DataLoader
dataloader_file = os.path.join(path+ f"{model_name}-finetuned-wikitext2", "test_dataloader_state.pt")
if not os.path.exists(dataloader_file):
    raise FileNotFoundError("Saved test dataloader state was not found.")

#saved_state = torch.load(dataloader_file, map_location="cpu")# unpicklig error: This error happens because PyTorch restricted its default file-loading settings to block non-weight data structures from executing untrusted code.
saved_state = torch.load(dataloader_file, map_location="cpu", weights_only=False)#fast fix of above error

# Rebuild the HF Dataset object from the saved dict
reconstructed_dataset = Dataset.from_dict(saved_state["dataset_dict"])
reconstructed_dataset.set_format("torch", columns=["input_ids", "token_type_ids", "attention_mask", "labels"])

# Rebuild the standard PyTorch Dataloader
test_dataloader = DataLoader(
    reconstructed_dataset,
    batch_size=saved_state["batch_size"],
    collate_fn=saved_state["collate_fn"],
    num_workers=saved_state["num_workers"],
    pin_memory=saved_state["pin_memory"],
    shuffle=False
)

# 3. Prepare components for the inference accelerator device context
model_infer, test_dataloader = accelerator.prepare(model_infer, test_dataloader)

# 4. Perform Inference Loop
model_infer.eval()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

DistilBertForMaskedLM(
  (activation): GELUActivation()
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0

In [ ]:
# Check if pad_token is already assigned; if not, set it to the eos_token
if tokenizer_infer.pad_token is None:
    tokenizer_infer.pad_token = tokenizer.eos_token

**Inference on test sentences**

1)Frozen Randoming masking in batch of test sentences

In [ ]:
len(test_dataloader)

69

Previously to avoid **DataCollatorForLanguageModeling also applies random masking with each evaluation and test**:-

1.) eliminate this source of randomness is to apply the **masking once on the whole test set, and then use the default data collator in 🤗 Transformers to collect the batches during evaluation**(validation set during training and test set in inference after loading the saved fine tuned model).

2.) Save the prepared test dataloader mapped with particular permutation state of mask_id assignment by **DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)** during **initial creation of train, validation and test dataloader during training loop setup**

In [ ]:
model_infer.eval()
torch.manual_seed(42)

from tqdm import tqdm
losses =[]


for step, batch in tqdm(enumerate(test_dataloader)):
  with torch.no_grad():
      outputs = model_infer(**batch)

  loss = outputs.loss
  losses.append(accelerator.gather(loss.repeat(batch_size)))


losses = torch.cat(losses)
print(losses)
losses = losses[: len(test_dataloader)]

test_loss = losses.mean().item()
print(f">>> Saved model: avg_test Loss: {test_loss}")

try:
    perplexity = math.exp(torch.mean(losses))
except OverflowError:
    perplexity = float("inf")

print(f">>> Saved model: Perplexity: {perplexity}")

69it [00:10,  6.32it/s]


tensor([1.8454, 1.8454, 1.8454,  ..., 2.2279, 2.2279, 2.2279], device='cuda:0')
>>> Saved model: avg_test Loss: 1.9583160877227783
>>> Saved model: Perplexity: 7.0873824807901995


In [ ]:
for batch in test_dataloader:

  print(type(batch))
  del batch["labels"]
  features = [dict(zip(batch, t)) for t in zip(*batch.values())]#convert dict of list blocks into list of indvl dict having values as indvl chunks for
  #corresponding keys

  #print(features)

  for block in features[:3]:

    for key in block.keys():
      block[key] =block[key].reshape(1, block_size)

    print(tokenizer_infer.decode(block["input_ids"]))

    #block =block.convert_to_tensors(tensor_type="pt")

    token_logits = model_infer(**block).logits #indivl inp_ids_pred_start

    # Find where the mask tokens are located
    mask_token_indices = torch.where(
        block["input_ids"] == tokenizer_infer.mask_token_id
    )[1]

    # Extract predictions for each mask index
    for idx in mask_token_indices:
      mask_logits = token_logits[0, idx, :]
      top_tokens = torch.topk(mask_logits, k=1, dim=-1).indices

      decoded_words = [tokenizer_infer.decode([token]) for token in top_tokens]

      block["input_ids"][0][idx] = top_tokens[0]#replace mask token_ids with topmost predicted ids for each masked index

      print(f"Predictions for mask at index {idx.item()}: {decoded_words}") #indivl inp_ids_pred_ends

    print(tokenizer_infer.decode(block["input_ids"]))# aggregate sentence with all topmost predicted tokens replaces
    print("*"*700)
  break

<class 'dict'>
['[CLS] [SEP] [CLS] = robert boult [MASK] = [SEP] [CLS] [SEP] [CLS] robert [MASK]ulter is an english film, television [MASK] theatre actor [MASK] he had a guest @ - @ starring role on the television series the bill in 2000. this was followed by a starring role in the play herons written by simon stephens, which [MASK] performed in 2001 at the royal court theatre. [MASK] had [MASK] guest [MASK] in the television [MASK] judge john deed in 2002. in 2004 boult [MASK] landed a role as " craig " in the episode " teddy \' s story " [MASK] habitat television [MASK] the long firm ; [MASK] starred alongside actors mark strong and derek jacobi [MASK]']
Predictions for mask at index 7: ['##er']
Predictions for mask at index 14: ['bo']
Predictions for mask at index 23: ['and']
Predictions for mask at index 26: ['.']
Predictions for mask at index 63: ['was']
Predictions for mask at index 73: ['he']
Predictions for mask at index 75: ['a']
Predictions for mask at index 77: ['appearance'

In [ ]:
for block in lm_datasets["test"][:3]["input_ids"]:
  print(tokenizer_infer.decode(block))
  print("*"*50)

[CLS] [SEP] [CLS] = robert boulter = [SEP] [CLS] [SEP] [CLS] robert boulter is an english film, television and theatre actor. he had a guest @ - @ starring role on the television series the bill in 2000. this was followed by a starring role in the play herons written by simon stephens, which was performed in 2001 at the royal court theatre. he had a guest role in the television series judge john deed in 2002. in 2004 boulter landed a role as " craig " in the episode " teddy ' s story " of the television series the long firm ; he starred alongside actors mark strong and derek jacobi.
**************************************************
he was cast in the 2005 theatre productions of the philip ridley play mercury fur, which was performed at the drum theatre in plymouth and the menier chocolate factory in london. he was directed by john tiffany and starred alongside ben whishaw, shane zaza, harry kent, fraser ayres, sophie stanton and dominic hall. [SEP] [CLS] in 2006, boulter starred along

2)part of sentnces/Inference on the random sentence

single_mask

In [ ]:
#text = "the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was performed at the drum theatre in [MASK]."
text = "This is a great [MASK]."

inputs = tokenizer_infer(text, return_tensors="pt").to(model_infer.device)# to bring input tensor at gpu (selected level)
print(inputs)
token_logits = model_infer(**inputs).logits

# Find the location of [MASK] and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer_infer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]
# Pick the [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=-1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer_infer.mask_token, tokenizer_infer.decode([token]))}'")

{'input_ids': tensor([[ 101, 2023, 2003, 1037, 2307,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}
'>>> This is a great idea.'
'>>> This is a great deal.'
'>>> This is a great success.'
'>>> This is a great adventure.'
'>>> This is a great relief.'


multiple_masks

In [ ]:
text = "the 2005 theatre productions of the Philip Ridley play Mercury Fur , which [MASK] performed at the drum theatre in [MASK]."
#text = "This is a great [MASK]."

inputs = tokenizer_infer(text, return_tensors="pt").to(model_infer.device)# to bring input tensor at gpu (selected level)
print(inputs)
token_logits = model_infer(**inputs).logits

# Find where the mask tokens are located
mask_token_indices = torch.where(
    inputs["input_ids"] == tokenizer_infer.mask_token_id
)[1]

# Extract predictions for each mask index
for idx in mask_token_indices:
  mask_logits = token_logits[0, idx, :]
  top_tokens = torch.topk(mask_logits, k=5, dim=-1).indices

  decoded_words = [tokenizer_infer.decode([token]) for token in top_tokens]
  print(f"Predictions for mask at index {idx.item()}: {decoded_words}")


{'input_ids': tensor([[  101,  1996,  2384,  3004,  5453,  1997,  1996,  5170, 20608,  2377,
          8714,  6519,  1010,  2029,   103,  2864,  2012,  1996,  6943,  3004,
          1999,   103,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Predictions for mask at index 14: ['was', 'were', 'also', 'is', 'he']
Predictions for mask at index 21: ['london', 'dublin', 'manchester', 'melbourne', 'edinburgh']


# Functions of Datacollator

*   **Common function(mlm =False or mlm= some fraction<=0.15)**

**DataCollatorForLanguageModeling** converts list of identical to something analogous to dictionaries of same keylist with values concatanation of individual dict chunks inside input list for given keys.

Apart from DataCollatorForLanguageModeling's function to PAD variable tokens to local batch_max_length and inserting -100 as tokens for PAD in input_ids, and creates labels on fly.

*   **Specific func(when mlm<=0.15 i.e. some fraction when collating for masked language model)**

Randomly assigns given fraction [MASK] in text and subsitutes input_ids tokens as **tokenizer.mask_token_id** and retains **labels token values at for mask positions, and assigns rest as -100**



